In [ ]:
!wget https://dumps.wikimedia.org/enwiki/latest/enwiki-latest-pages-articles.xml.bz2

In [ ]:
!pip install pyserini wikiextractor

In [ ]:
!python -m wikiextractor.WikiExtractor enwiki-latest-pages-articles.xml.bz2 --json --output wiki_output

In [ ]:
!conda install -y -c conda-forge openjdk=21

In [4]:
!find wiki_output -type f ! -name "*.json" -exec mv {} {}.json \;

In [ ]:
import os
import json
from concurrent.futures import ProcessPoolExecutor

def fix_json_keys(filepath):
    """Reads a file line-by-line, changes 'text' to 'contents', and overwrites the file."""
    with open(filepath, 'r', encoding='utf-8') as f:
        lines = f.readlines()
        
    with open(filepath, 'w', encoding='utf-8') as f:
        for line in lines:
            try:
                # Load the JSON line
                doc = json.loads(line)
                
                # If 'text' exists, rename it to 'contents'
                if 'text' in doc:
                    doc['contents'] = doc.pop('text')
                    
                # Write the fixed JSON back to the file
                f.write(json.dumps(doc, ensure_ascii=False) + '\n')
            except json.JSONDecodeError:
                # If a line fails to parse, just write it back as-is
                f.write(line) 

# 1. Find all the .json files in your wiki_output directory
folder_path = 'wiki_output'
all_files = []
for root, dirs, files in os.walk(folder_path):
    for file in files:
        if file.endswith('.json'):
            all_files.append(os.path.join(root, file))

print(f"Found {len(all_files)} files. Starting conversion across 8 CPU cores...")

# 2. Process them in parallel
with ProcessPoolExecutor(max_workers=8) as executor:
    executor.map(fix_json_keys, all_files)

print("All files fixed successfully! You are ready to index.")

In [ ]:
!_JAVA_OPTIONS="-Xmx32G" python -m pyserini.index.lucene \
  --collection JsonCollection \
  --input wiki_output \
  --index wikipedia_bm25_index \
  --generator DefaultLuceneDocumentGenerator \
  --threads 8 \
  --storePositions --storeDocvectors --storeRaw

In [ ]:
import os
import json

# 1. Inject a fake API key to bypass Pyserini's startup check
os.environ["OPENAI_API_KEY"] = "sk-dummy-key-to-bypass-error"

# 2. NOW you can safely import Pyserini
from pyserini.search.lucene import LuceneSearcher

# 3. Load your brand new index
print("Loading index...")
searcher = LuceneSearcher('wikipedia_bm25_index')
searcher.set_bm25(k1=0.9, b=0.4)
# 4. Type whatever you want to search for here
query = "bobcat Bobcat animal bobcat lynx rufus wildcat North American bobcat bobcat animal bobcat wildlife bobcat facts bobcat habitat bobcat behaviort"
print(f"Searching for: '{query}'\n")

# 5. Fetch the top 3 results
hits = searcher.search(query, k=20)

# 6. Display the results
for i in range(len(hits)):
    doc_id = hits[i].docid
    score = hits[i].score
    
    # Grab the actual text from the index
    doc = searcher.doc(doc_id)
    content = json.loads(doc.raw())
    
    print(f"Rank {i+1} | Score: {score:.4f} | Title: {content['title']}")
    print(f"Preview: {content['contents'][:250]}...\n")
    print("-" * 60)